# Recurrent Neural Networks
In this exercise, we will implement a simple one-layer recurrent neural network. We will use the formula for an [Elman RNN](https://en.wikipedia.org/wiki/Recurrent_neural_network#Elman_networks_and_Jordan_networks), one of the most basic and classical RNNs. The hidden state update and output at time $t$ are defined like this:

$$
\begin{align}
h_t &= \tanh(W_h x_t + U_h h_{t-1} + b_h) \\
y_t &= \tanh(W_y h_t + b_y)
\end{align}
$$

In [10]:
import torch
import torch.nn as nn

We start by defining the RNN as a subclass of `nn.Module`. The network's parameters are created in the `__init__` method. Use `input_dim`, `hidden_dim` and `output_dim` as arguments that define the dimensionality of the input/hidden/output vectors. Define your parameters as `nn.Parameter` with the appropriate dimensions. The documentation of `torch.nn` can be found [here](https://pytorch.org/docs/stable/nn.html).

In [11]:
class RNN(nn.Module):
    
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        self.W_xh = nn.Parameter(torch.zeros(hidden_dim, input_dim))
        self.W_hh = nn.Parameter(torch.zeros(hidden_dim, hidden_dim))
        self.W_hy = nn.Parameter(torch.zeros(output_dim, hidden_dim))
        
        self.b_h = nn.Parameter(torch.zeros(hidden_dim))
        self.b_y = nn.Parameter(torch.zeros(output_dim))
        

Add a function `reset_parameters` that initializes your parameters. Pick a suitable distribution from [nn.init](https://pytorch.org/docs/stable/nn.init.html).

In [12]:
def reset_parameters(self):
    for weight in self.parameters():
        nn.init.normal_(weight, 0, 1)

RNN.reset_parameters = reset_parameters

Add a `forward` function that takes an input and a starting hidden state $h_{t-1}$ and returns the updated hidden state $h_t$ and output $y$ as outputs. The initial hidden state $h_0$ can be initialized randomly/to all zeros.

In [13]:
def forward(self, x, hidden_state):
    h_t = torch.tanh(self.W_xh @ x + self.W_hh @ hidden_state + self.b_h)
    y_t = torch.tanh(self.W_hy @ h_t + self.b_y)
    return y_t, h_t

RNN.forward = forward

Test your RNN with a single input.

In [14]:
input_dim = 5
hidden_dim = 20
output_dim = 10

rnn = RNN(input_dim, hidden_dim, output_dim)
rnn.reset_parameters()

x = torch.randn(input_dim)
h0 = torch.zeros(hidden_dim)

y, new_hidden_state = rnn(x, h0)

print(y, new_hidden_state)

tensor([ 0.6493,  0.9991, -0.9829, -1.0000, -0.9658,  0.9852,  1.0000, -1.0000,
        -0.3858, -0.8004], grad_fn=<TanhBackward0>) tensor([ 0.9998, -0.9697, -0.3026,  0.5510, -0.6076,  0.9962,  0.9162,  0.9979,
         0.9929, -0.9990, -0.9999,  0.9592,  0.8781,  0.9987,  0.9016,  0.9870,
        -0.9995, -0.9242,  1.0000, -0.9369], grad_fn=<TanhBackward0>)


Now create an input sequence and run it through your RNN.

In [15]:
seq_length = 4
inputs = [torch.randn(input_dim) for _ in range(seq_length)]
hidden_state = torch.zeros(hidden_dim)
outputs = []

for x in inputs:
    y, hidden_state = rnn(x, hidden_state)
    outputs.append(y)
    
print(outputs)

[tensor([ 0.9147, -0.9962,  0.9280,  1.0000, -1.0000, -0.9996, -1.0000, -0.6711,
        -0.7086,  0.3465], grad_fn=<TanhBackward0>), tensor([-1.0000, -1.0000, -0.9998,  1.0000,  0.3663, -0.9975, -0.5554, -0.9951,
         1.0000,  0.9048], grad_fn=<TanhBackward0>), tensor([-0.9295,  0.9976,  0.8760, -0.7332,  0.9999,  1.0000,  0.9188,  0.9417,
         0.9999,  0.0238], grad_fn=<TanhBackward0>), tensor([ 0.9998,  0.6637, -0.8268, -0.9039, -0.7576, -0.6010,  0.9998,  0.0243,
        -0.6303, -0.9751], grad_fn=<TanhBackward0>)]


The final hidden state encodes all the information present in the input sequence. It can be used as a feature for classification, or to initialize a decoder RNN to do translation, for example.

Now look at PyTorch's documentation for the [`nn.RNN`](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html) and the [`nn.RNNCell`](https://pytorch.org/docs/stable/generated/torch.nn.RNNCell.html) classes. What is the difference between the two? What is the difference to the definition from Wikipedia we used above? Run your input sequence through both the `nn.RNN` and the `nn.RNNCell`.

In [16]:
rnn = nn.RNNCell(input_dim, hidden_dim)
input = torch.randn(seq_length, input_dim)
hx = torch.zeros(hidden_dim)
output = []
for i in range(seq_length):
    hx = rnn(input[i], hx)
    output.append(hx)
    
print(len(output))
print(output[0].shape)

4
torch.Size([20])


In [18]:
num_layers = 1

rnn = nn.RNN(input_dim, hidden_dim, num_layers)
input = torch.randn(seq_length, input_dim)
h0 = torch.zeros(num_layers, hidden_dim)
output, hn = rnn(input, h0)

print(output.shape)

torch.Size([4, 20])
